In [2]:
import xarray as xr
import numpy as np
import numpy.ma as ma
from datetime import datetime as dt
from datetime import timedelta

import matplotlib.pyplot as plt
import cartopy.crs as ccrs

import glob, sys, os, re
from pprint import pprint
import subprocess

In [29]:
sim_dir = "/lustre/fsn1/projects/rech/lsv/ubd61nl/ObedSaba_RECONCILE_MesoNH/production/WPAGG_2km"
fig_dir = '../figures'

In [4]:
! ls /lustre/fsn1/projects/rech/lsv/ubd61nl/ObedSaba_RECONCILE_MesoNH/production/WPAGG_2km

OBSOLETE_HELD_JOBS_20260912.txt  t0_20160910_00       TEGE_START_JOBS.txt
p12_20160910_12			 TEGE_SEG02_JOBS.txt
p6_20160910_06			 TEGE_SEG03_JOBS.txt


In [5]:
# Files currently available (21/09/2026): 
# pour p12_20160910_12 :
# - seg01_12h : pas 001 à 036 -- correct
# - seg02_12h : pas 001 à 026 -- OLR corrompu par le restart
# - seg01_12h_tege : pas 001 à 036 -- correct
# - seg02_12h_tege : pas 001 à 036 -- correct
# pour p6_20160910_06 :
# - seg01_12h : pas 001 à 035 -- correct mais incomplet
# - seg01_12h_tege : pas 001 à 036 -- correct
# - seg02_12h_tege : pas 001 à 036 -- correct
# pour t0_20160910_00 :
# - seg01_12h : pas 001 à 036 -- correct
# - seg02_12h : pas 001 à 025 -- OLR corrompu par le restart
# - seg01_12h_tege : pas 001 à 036 -- correct
# - seg02_12h_tege : pas 001 à 036 -- correct

In [6]:
# Define simulation subfolders
sim_names = [f for f in os.listdir(sim_dir) if re.match("(.*)2016(.*)",f) ]
sim_names.sort(reverse=True)
print(sim_names)

['t0_20160910_00', 'p6_20160910_06', 'p12_20160910_12']


In [7]:
for simname in sim_names:

    print(os.path.join(sim_dir,simname,"*"))
    # pprint(glob.glob(os.path.join(sim_dir,simname,'*')))
    # pprint(os.listdir(os.path.join(sim_dir,simname,'seg0?_*h')))
    file_list = glob.glob(os.path.join(sim_dir,simname,'seg0?_*h_tege','*.OUT.*.nc'))
    file_list.sort()
    pprint(len(file_list))
    print()

/lustre/fsn1/projects/rech/lsv/ubd61nl/ObedSaba_RECONCILE_MesoNH/production/WPAGG_2km/t0_20160910_00/*
72

/lustre/fsn1/projects/rech/lsv/ubd61nl/ObedSaba_RECONCILE_MesoNH/production/WPAGG_2km/p6_20160910_06/*
72

/lustre/fsn1/projects/rech/lsv/ubd61nl/ObedSaba_RECONCILE_MesoNH/production/WPAGG_2km/p12_20160910_12/*
72



In [33]:
def getOLRFromTb(Tb):
    """Compute OLR estimate in W/m2 from Tb in K, using the conversion formula from Ohring & Gruber (1984),
    with coefficients reported in Yang & Slingo (2001), also used in Feng et al. (2024):

    OLR = sigma * Tf^4

    where

    Tf = Tb * (a + b Tb)

    with a = 1.228 and b = -1.106e-3 K-1.
    """

    sigma = 5.67e-8 # W/m2/K4
    a = 1.228 # no unit
    b = -1.106e-3 # K-1
    
    return sigma * (a * Tb + b * Tb**2)**4

def getTbFromOLR(OLR):
    """Compute Tb estimate in K from OLR in W/m2, by reverting the conversion formula from Ohring & Gruber (1984),
    with coefficients reported in Yang & Slingo (2001), also used in Feng et al. (2024):

    OLR = sigma * Tf^4

    where

    Tf = Tb * (a + b Tb)

    with a = 1.228 and b = -1.106e-3 K-1.
    """

    sigma = 5.67e-8 # W/m2/K4
    a = 1.228 # no unit
    b = -1.106e-3 # K-1
    
    return (-a + np.sqrt(a**2 + 4 * b* (OLR/sigma)**(1/4)))/(2*b)

In [14]:
# Load all sims at a specific time

def getFileAtTime(target_time,time_ref=dt(2016,9,10),lag_h=0):
    """For a given lag (in hours) at the start of the simulation,
    get the filename of the right simulation at that given time.
    """
    
    i_lag = np.where(lags_h == lag_h)[0][0]
    
    #-- get corresponding time file
    # time difference
    delta_t = target_time - (time_ref+timedelta(seconds=lag_h*3600))
    # find corresponding subfolder
    i_seg = int((delta_t.total_seconds()/3600/Dt - 1) / (12/Dt) ) +1 # segment number
    subfolder = "seg0%d_12h_tege"%i_seg
    # build corresponding filename
    i_t_seg = int((delta_t.total_seconds()/3600/Dt - 1) % int(12/Dt) + 1)
    filename = "FBC01.1.MNH0%d.OUT.%s.nc"%(i_seg,str(i_t_seg).zfill(3))
    # full path
    path = os.path.join(sim_dir,sim_names[i_lag],subfolder,filename)

    return path


In [83]:
##-- Define lags between simulations

lags_h = np.array([0,6,12])

lag_6h = timedelta(seconds=6*3600)
lag_12h = timedelta(seconds=12*3600)
time_ref = dt(2016,9,10)
Dt = 1/3 # h

# target_time = dt(2016,9,10,13,00)
# target_time = dt(2016,9,10,14,00)

##-- Define lags between simulations

lags_h = np.array([0,6,12])

lag_6h = timedelta(seconds=6*3600)
lag_12h = timedelta(seconds=12*3600)
time_ref = dt(2016,9,10)
Dt = 1/3 # h

# target_time = dt(2016,9,10,13,00)
# target_time = dt(2016,9,10,14,00)
target_time = dt(2016,9,10,20,00)

##-- Define bounds in OLR
vmin = 70
vmax = 170

In [84]:
##-- Check that the files retrieved at a given time seem correct

for i_lag in range(3):
    
    lag_h = float(lags_h[i_lag])
    
    path_at_time = getFileAtTime(target_time,lag_h=lag_h)
    print(path_at_time)


/lustre/fsn1/projects/rech/lsv/ubd61nl/ObedSaba_RECONCILE_MesoNH/production/WPAGG_2km/t0_20160910_00/seg02_12h_tege/FBC01.1.MNH02.OUT.024.nc
/lustre/fsn1/projects/rech/lsv/ubd61nl/ObedSaba_RECONCILE_MesoNH/production/WPAGG_2km/p6_20160910_06/seg02_12h_tege/FBC01.1.MNH02.OUT.006.nc
/lustre/fsn1/projects/rech/lsv/ubd61nl/ObedSaba_RECONCILE_MesoNH/production/WPAGG_2km/p12_20160910_12/seg01_12h_tege/FBC01.1.MNH01.OUT.024.nc


The file identification seems to work well

In [85]:
##-- Get Meso-NH data and coords

data_lag0 = xr.open_dataset(getFileAtTime(target_time,lag_h=0))
data_lag6 = xr.open_dataset(getFileAtTime(target_time,lag_h=6))
data_lag12 = xr.open_dataset(getFileAtTime(target_time,lag_h=12))

##- Cap data in OLR range
RLUT_lag0 = data_lag0.RLUT_INST[0].values
RLUT_lag0[RLUT_lag0 > vmax] = np.nan
RLUT_lag0[RLUT_lag0 < vmin] = np.nan
RLUT_lag6 = data_lag6.RLUT_INST[0].values
RLUT_lag6[RLUT_lag6 > vmax] = np.nan
RLUT_lag6[RLUT_lag6 < vmin] = np.nan
RLUT_lag12 = data_lag12.RLUT_INST[0].values
RLUT_lag12[RLUT_lag12 > vmax] = np.nan
RLUT_lag12[RLUT_lag12 < vmin] = np.nan

##-- Domain data
lon_array = data_lag0.longitude.values
lat_array = data_lag0.latitude.values
longitude = data_lag0.longitude[0].values
latitude = data_lag0.latitude[:,0].values
extent = [longitude[0],longitude[-1],latitude[0],latitude[-1]]

In [86]:
##-- Load observed Tb
#
# Data copied to jeanzay from spirit1:/bdd/GEOgrid_coldcloud/
#

# parent_dir = '/lustre/fswork/projects/rech/lsv/uae56hc/observations/GEOgrid_coldcloud/HIMAWARI+1407'
parent_dir = '/lustre/fswork/projects/rech/lsv/uae56hc/observations/GEOgrid_coldcloud/GOES-W-1350'
data_dir = os.path.join(parent_dir,str(target_time.year),target_time.strftime("%Y_%m_%d"))
# file_name = 'GEO_L1C-HIMA08_%s-00_G_IR112_004_V1.1.nc'%target_time.strftime("%Y-%m-%dT%H-%M")
file_name = 'GEO_L1C-GOES15_%s-00_G_IR107_004_V1.1.nc'%target_time.strftime("%Y-%m-%dT%H-%M")
file_path = os.path.join(data_dir,file_name)

data_obs = xr.open_dataset(file_path)

/linkhome/rech/genlmd01/uae56hc/.conda/envs/pyLMD/lib/python3.14/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'Harmonized_irBT' has multiple fill values {np.int32(-99800), np.int32(-999)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


In [87]:
##-- Format observations to display

# Select the same region as in Meso-NH
lon_slice = slice(extent[0]+360,extent[1]+360)
lat_slice = slice(extent[3],extent[2])
longitude_obs = data_obs.longitude.sel(longitude=lon_slice)
latitude_obs = data_obs.latitude.sel(latitude=lat_slice)
lon_array_obs, lat_array_obs = np.meshgrid(longitude_obs,latitude_obs)

# Convert observed Tb to OLR estimate (W/m2)
data_OLR = getOLRFromTb(data_obs.Harmonized_irBT.sel(longitude=lon_slice,latitude=lat_slice)).values

# Cap data in range
data_OLR[data_OLR > vmax] = np.nan
data_OLR[data_OLR < vmin] = np.nan

In [88]:
longitude_obs

<xarray.DataArray 'longitude' (longitude: 552)> Size: 2kB
array([183.48, 183.52, 183.56, ..., 205.44, 205.48, 205.52],
      shape=(552,), dtype=float32)
Coordinates:
  * longitude  (longitude) float32 2kB 183.5 183.5 183.6 ... 205.4 205.5 205.5
Attributes:
    units:       degrees_east
    long_name:   longitude
    FlagDegEst:  1

In [89]:
195-360

-165

In [ ]:
def showSubplot(lon_array, lat_array, data_var, data_crs, extent, vmin=70, vmax=170):
    
    ax.set_extent(extent, crs=data_crs)
    # data
    # mask = np.logical_and(data_var > vmax, data_var < vmin)
    pcm = ax.pcolormesh(lon_array, lat_array, 
                        # ma.masked_array(data_var,mask),
                        data_var,
                        transform=data_crs,
                        vmin = vmin,
                        vmax = vmax)
    # Côtes
    ax.coastlines(resolution='10m', color='black', linewidth=0.8)
    # grille
    gl = ax.gridlines(crs=data_crs, draw_labels=True,
                       color='black', linewidth=0.2, linestyle='-')
    gl.top_labels = False
    gl.right_labels = False
    # colorbar
    cb = plt.colorbar(pcm, ax=ax, orientation='vertical', shrink=0.8)
    cb.ax.set_ylabel('OLR (W/m2)')


data_crs = ccrs.PlateCarree()  # projection de vos données lon/lat

fig, axs = plt.subplots(ncols=2, nrows=2, figsize=(10, 10),
                         subplot_kw={'projection': data_crs})
axs = axs.flatten()

##-- (0) observations
ax = axs[0]
showSubplot(lon_array_obs, lat_array_obs, data_OLR[0], data_crs, extent)
ax.set_title('GOES-W')

##-- (1) simulations depuis t=0
ax = axs[1]
showSubplot(lon_array, lat_array, RLUT_lag0, data_crs, extent)
ax.set_title('$t_0 = $%s'%(dt.strftime(time_ref,"%Y-%m-%dT%H:%M")))

##-- (2) simulations depuis t=6h
ax = axs[2]
showSubplot(lon_array, lat_array, RLUT_lag6, data_crs, extent)
ax.set_title('$t_0 = $%s'%(dt.strftime(time_ref+lag_6h,"%Y-%m-%dT%H:%M")))

##-- (3) simulations depuis t=12h
ax = axs[3]
showSubplot(lon_array, lat_array, RLUT_lag12, data_crs, extent)
ax.set_title('$t_0 = $%s'%(dt.strftime(time_ref+lag_12h,"%Y-%m-%dT%H:%M")))

save_path = os.path.join(fig_dir,'RC5_comparison_t0_%s.png'%(dt.strftime(target_time,"%Y%m%dT%H%M")))
plt.savefig(save_path,bbox_inches='tight')